This notebook is used to document all operations required to perform a full cycle of the CPD injection–recovery experiment for a given gap in a protoplanetary disk. Our goal is to assess the detectability of circumplanetary disks within these gaps. The content compiled here will serve as the basis for the input .py files executed on the cluster. Ultimately, the aim is to obtain the recovery–fraction curve for each gap and planet-kink location, providing further constraints for the forward-modelling predictions of CPD properties.

# Yesterday's Problem (11/17)

Problem 1 to be addressed: 

1. The SPW shape of the AA_Tau original measurement set is shown as the following plot.

```python
from casatools import table
import numpy as np

out_ms = r"d/mnt/exoALMA_disk_data/data/AA_Tau_time_ave_continuum.ms"

spw_tb = table()
spw_tb.open(out_ms + "/SPECTRAL_WINDOW")

# Number of channels for each SPW
nchan = spw_tb.getcol("NUM_CHAN")

# REF_FREQUENCY column = central / reference frequency of SPW (Hz)
ref_freq = spw_tb.getcol("REF_FREQUENCY")

spw_tb.close()

# Print summary
print("SPW | nchan | ref_freq (GHz)")
for spw in range(len(nchan)):
    print(f"{spw:3d} | {nchan[spw]:5d} | {ref_freq[spw]/1e9:10.6f}")
```

```python
#-----------InjectLoop--------------#
# load the visibility data
dat = np.load('data/'+target+'_data.vis.npz')
u, v, vis, wgt = dat['u'], dat['v'], dat['Vis'], dat['Wgt']
# vis.shape = (Nrows,)


#-------------IMPORTMS----------------#
# load the model visibilities
mdl = (np.load(modelfile+'.npz'))['V']
# replace with the model visibilities (equal in both polarizations)
data[:, :, unflagged] = mdl
# data.shape = (Nrows, Npol, Nchan)
```


In my continuum MS, the SPW table lists multiple SPWs with various channel counts, but the DATA column in the MAIN table has shape (2, 1, 2,456,010). Does this mean that only the SPWs with a single averaged channel are actually used to construct this MS, and the others exist only as unused metadata? In other words, is the MS effectively single-channel because only those one-channel SPWs appear in the MAIN table? And if so, does that imply that during the import-MS step I can only inject the model visibilities into rows whose shapes match this (2, 1, Nrows) structure?

Also, looking into the way that frank recommends using uvplot in CASA to export an MS into uv tables, the function can only handle MS tables where all SPWs have the same number of channels. From exoALMA IV, they also mention in Section 2 that the visibilities in the continuum spectral windows were spectrally averaged down to one channel for the continuum analysis.

So I think it’s okay if I don’t work on the original MS but instead use a split version with spectral averaging, since I’m not sure whether the multi-channel MS can even be converted into the .npz format (in my current workflow, vis becomes a 1D array after loading with NumPy). CASA also throws an error when trying to extract the DATA column from an MS with mixed channel counts: "RuntimeError: ArrayColumn::getColumn cannot be done for column DATA; the array shapes"


In [8]:
import numpy as np

# Specific to CPD injection pipeline
target = 'AA_Tau'
# Use raw string to avoid escape sequence issues
dat = np.load(rf'D:\exoALMA_disk_data\data\{target}_time_ave_continuum.vis.npz')

print("Arrays in NPZ file:", dat.files)
print("\nArray shapes:")
for array_name in dat.files:
    print(f"  {array_name}: {dat[array_name].shape}")

print(f"\nVisibility data for {target}:")
print(f"Total visibility points: {len(dat['Vis'])}")

vis = dat['Vis']
print("vis dtype:", vis.dtype)

Arrays in NPZ file: ['u', 'v', 'Vis', 'Wgt']

Array shapes:
  u: (2456010,)
  v: (2456010,)
  Vis: (2456010,)
  Wgt: (2456010,)

Visibility data for AA_Tau:
Total visibility points: 2456010
vis dtype: complex128


Another thing I will try to do, is to do the altered export_ms in one ms file and inspect the npz file .

```python
cd /mnt/d/CPD_MPIA_Injection_Recovery_trial_fBf/DSHARP_source_code
msfile = "d/mnt/exoALMA_disk_data/data/AA_Tau_time_ave_continuum.ms"
outfile = "AA_Tau_spec_avg_data.vis.npz"

exec(open("export_ms_NONspectralavg.py").read())

```

#

# Yesterday's Work  (11/18)

1. Duplicate `/data` folder → `D:\exoALMA_disk_data\measurement_set`   
2. Split measurement set → spectrally-averaged measurement set
3. Create UV table from measurement set → `.npz` format, using uvplot inside CASA
```python
# CASA split function for all disks
cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/AA_Tau_robust2_0_gap0
/usr/local/bin/CASA/casa-6.6.1-17-pipeline-2024.1.0.8/bin/casa
execfile('spavg_ms.py', globals()) # to run spectral averaging

execfile('check_flags.py', globals())  # Check flagged data fraction
execfile('check_spw_channels.py', globals())  # final check for SPW

execfile('ms_to_npz.py', globals()) # convert the spavg ms to npz for fra

execfile('AA_Tau_robust2_0_gap0_injectloop.py', globals()) 

execfile('ms_to_npz2.py', globals())  # correct for the u,v unit.

# try to run injection again:
python AA_tau_1core_injectloop.py 
5. run an injection loop for one disk , using 3 flux bin
6. run the custom masking scripts

# Yesterday's Work (11/19)

1. Rearrange the vis.txt back into the exoALMA_data folder (done)

2. Run the new frank injection script in Ubuntu (no need to be inside casa)

```python
cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/AA_Tau_robust2_0_gap0
python AA_Tau_robust2_0_gap0_injectloop.py
python AA_tau_Prev_injectloop.py
```
 PROBLEM: Weird, it does not work and show errror in computing negative power spectrum value. Even if i go back to use the previous version and the previsious visibility table. Maybe sth is wrong with the dictionary

 -> try to run the original script with parallel processing
 python AA_tau_1core_injectloop.py
3. Check the r and azimuthal values in the mpars.txt

4. Apply the custom masking file
  create initial mask -> opent the mask fits to run custom mask funciton which isolates the gap region


In [ ]:
# print the pkl to compare

# Today's Work (11/20)

<font color="green">Manually move the mpars.txt into the injection folder</font> before prepimaging


Masking for the imageloop 
Note: Might need to rename the J ones
In DSHARP source code

1. Put preimaging, mask_to_casa, preimaging_casa, custom_mask in the same subdirectory (DSHARP_source_code)
2. check the entire preimaging.py

```python
cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/AA_Tau_robust2_0_gap0
python prepimaging.py
```

issue: disk_dictionaryr2_0 not found again, to fix,  ADD SYSTEM PATH TO EVERY FILE RUN IN CASA

```python
import sys, os
sys.path.append(os.getcwd())
```

3. Open Carta to inspect the fits
```python
./carta-4.0-x86_64.AppImage   
```



# Today's work (11/21) 

<font color="green">But the scariest part!!!!! IMAGELOOPPP </font>

1. Run Imageloop

```python
 cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/AA_Tau_robust2_0_gap0
/usr/local/bin/CASA/casa-6.6.1-17-pipeline-2024.1.0.8/bin/casa

 execfile("AA_Tau_robust2_0_gap0_imageloop.py", globals())
 execfile("AA_Tau_robust2_0_gap0_imageloop_rerun.py", globals())
 exec(compile(open("AA_Tau_robust2_0_gap0_imageloop.py", "rb").read(),"AA_Tau_robust2_0_gap0_imageloop.py", 'exec')) 


 python recover_loop.py
 python assess_recovery.py

#### Today's work (pipeline)





```python

#### Inject

(change disk name in injectloop.py)


cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/J1852_robust2_0_gap0

python J1852_1core_injectloop.py


(Move the .txt to injection folder)

#### Pre-imaging to create psf and sumwt   (Change disk name in preimaign.py)


cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/J1852_robust2_0_gap0
python prepimaging.py

#### Imaging 


 cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/J1852_robust2_0_gap0
/usr/local/bin/CASA/casa-6.6.1-17-pipeline-2024.1.0.8/bin/casa

 execfile("J1852_robust2_0_gap0_imageloop.py", globals())


#### Recovery

(Change recover_loop and assess_recovery disk name, hdu disk name, imdir disk name )

 cd /mnt/d/CPD_MPIA/Injection_Recovery_trial_fBf/J1852_robust2_0_gap0
python recover_loop.py
python assess_recovery.py

```

In [ ]:
Today's work

## Tooday's Work (1/30/2026)

```python

# Inject loop (Change disk name and gap ix)
# Pre- Imaging  (change disk name)
# Tonight move the stuff inside

```python
# Download
cd /nexus/posix0/MIA-astro-env/myben/vawelke
mkdir -p Downloads
cd Downloads

wget https://casa.nrao.edu/download/distro/casa-pipeline/release/linux/casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8.tar.xz

# Extract
cd /nexus/posix0/MIA-astro-env/myben/vawelke
mkdir -p software
cd software

tar -xf ../Downloads/casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8.tar.xz
# to show progress
tar -xvf ../Downloads/casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8.tar.xz

# Find the extracted directory name (now you're IN software/, so no "software/" prefix)
ls casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8/bin/casa

# Run + check version (same - you're already in software/)
casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8/bin/casa --pipeline --version
casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8/bin/casa --pipeline


# 1. Create the data directory
mkdir -p /nexus/posix0/MIA-astro-env/myben/vawelke/casa_data

# 2. Create config file in your project space (not home!)
echo "measurespath='/nexus/posix0/MIA-astro-env/myben/vawelke/casa_data'" > /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py

# 3. Verify the config file
cat /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py

# 4. Start CASA with custom config location
/nexus/posix0/MIA-astro-env/myben/vawelke/software/casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8/bin/casa --pipeline --configfile /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py

```


Remember with this structure the job script must look like

```python
#!/bin/bash
#SBATCH --job-name=casa_job
#SBATCH --time=02:00:00
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G

export PATH="/nexus/posix0/MIA-astro-env/myben/vawelke/software/casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8/bin:$PATH"
export CASA_CONFIG=/nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py

casa --pipeline --configfile $CASA_CONFIG -c your_script.py
```

### Results of installing casa data
```python
[vawelke@Node-01 software]$ mkdir -p /nexus/posix0/MIA-astro-env/myben/vawelke/casa_data
[vawelke@Node-01 software]$ echo "measurespath='/nexus/posix0/MIA-astro-env/myben/vawelke/casa_data'" > /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py
[vawelke@Node-01 software]$ cat /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py
measurespath='/nexus/posix0/MIA-astro-env/myben/vawelke/casa_data'
[vawelke@Node-01 software]$ /nexus/posix0/MIA-astro-env/myben/vawelke/software/casa-6.6.6-17-pipeline-2025.1.0.35-py3.10.el8/bin/casa --pipeline --configfile /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py

Using user configuration file /nexus/posix0/MIA-astro-env/myben/vawelke/casa_config.py

IPython 8.26.0 -- An enhanced Interactive Python.

No event loop hook running.
pull_data using version casarundata-2025.12.12-1.tar.gz, acquiring the lock ...
downloading casarundata contents to /nexus/posix0/MIA-astro-env/myben/vawelke/casa_data (336M) ... done
casarundata installed casarundata-2025.12.12-1.tar.gz at /nexus/posix0/MIA-astro-env/myben/vawelke/casa_data
measures_update ... acquiring the lock ...
  ... finding available measures at www.astron.nl ...
  ... downloading WSRT_Measures_20260131-160001.ztar from ASTRON server to /nexus/posix0/MIA-astro-env/myben/vawelke/casa_data ...
  ... measures data updated at /nexus/posix0/MIA-astro-env/myben/vawelke/casa_data
debug1: client_input_channel_open: ctype x11 rchan 3 win 65536 max 16384
debug1: client_request_x11: request from ::1 37792
debug1: channel 1: new x11-connection [x11] (inactive timeout: 0)
debug1: confirm x11
2026-02-01 12:37:15 INFO: Environment is not MPI enabled. Pipeline operating in single host mode
2026-02-01 12:37:20 INFO: Pipeline version 2025.1.0.35 running on Node-01
2026-02-01 12:37:20 INFO: Host environment:
        CPU: AMD EPYC 9454 48-Core Processor (physical cores: 96, logical cores: 192)
        Memory: 1416.8 GiB RAM, unknown swap
        OS: AlmaLinux 9.7 (Moss Jungle Cat)
        cgroup limits: N/A of 192 CPU cores, memory limits=N/A
        ulimit limits: CPU time=N/A, memory=N/A, files=1024
2026-02-01 12:37:20 INFO: Environment as detected by CASA:
        CPUs reported by CASA: 192 cores, max 192 OpenMP threads
        Available memory: 1416.8 GiB
2026-02-01 12:37:20 INFO: Initializing cli...
2026-02-01 12:37:20 INFO: Loaded Pipeline commands from package: h
2026-02-01 12:37:20 INFO: Loaded Pipeline commands from package: hif
2026-02-01 12:37:20 INFO: Loaded Pipeline commands from package: hifa
2026-02-01 12:37:20 INFO: Loaded Pipeline commands from package: hifv
2026-02-01 12:37:20 INFO: Loaded Pipeline commands from package: hsd
2026-02-01 12:37:20 INFO: Loaded Pipeline commands from package: hsdn
CASA 6.6.6.17 -- Common Astronomy Software Applications [6.6.6.17]

CASA <1>:
```

# Move dataset to directory

Pys in each folder
1. DSHARP code: inject_CPD, 
2. diskdictionaryr2_0 
3. dat = np.load('/nexus/posix0/MIA-astro-env/myben/vawelke/exoALMA_disk_data/measurement_set_spavg/npz/' + target + '_time_ave_continuum_spavg_lambda.vis.npz')
4. 

### Move  stuff
#### ms, npz, folder script for each disk/gap, source code 


```python

# My Ubuntu path: \\wsl.localhost\Ubuntu\home\vrice
# or log in Ubuntu and type code . to open vs code interface


# Create the destination folders
mkdir -p /nexus/posix0/MIA-astro-env/myben/vawelke/exoALMA_disk_data/measurement_set_spavg/npz


# Outside the node, in ubuntu
# Move only the measurement sets
rsync -av --progress --partial --append-verify\
  /mnt/d/exoALMA_disk_data/measurement_set_spavg/*.ms \
  astronode1:/nexus/posix0/MIA-astro-env/myben/vawelke/exoALMA_disk_data/measurement_set_spavg/



# move the npz files
rsync -av --progress \
  /mnt/d/exoALMA_disk_data/measurement_set_spavg/npz/ \
  astronode1:/nexus/posix0/MIA-astro-env/myben/vawelke/exoALMA_disk_data/measurement_set_spavg/npz/



# Check 
ls /nexus/posix0/MIA-astro-env/myben/vawelke/exoALMA_disk_data/measurement_set_spavg






# (02/02)   Actually runnning stuff

### Move script inside


```python
# move the folders
ssh astronode1 "mkdir -p /nexus/posix0/MIA-astro-env/myben/vawelke/inj_rev/J1852_gap0"

rsync -av --progress \
  /mnt/d/CPD_MPIA/HPC_scripts/J1852_gap0/ \
  astronode1:/nexus/posix0/MIA-astro-env/myben/vawelke/inj_rev/J1852_gap0/

ssh astronode1 "mkdir -p /nexus/posix0/MIA-astro-env/myben/vawelke/Source_codes"

rsync -av --progress \
  /mnt/d/CPD_MPIA/HPC_scripts/Source_codes/ \
  astronode1:/nexus/posix0/MIA-astro-env/myben/vawelke/Source_codes/

```

### trial run

```python
# 1. Copy and paste J1852_gap0 into J1852_gap0_trial1
cp -r J1852_gap0 J1852_gap0_trial1
# 2.  Change the flux to 3 values per flux bin
nano J1852_gap0_injectloop.py

# 3. Create the nano bash
nano trial1.sh
sbatch trial1.sh

# 4. run 


1. Move the folder of scripts into HPC
2. Create some simple slurm to run it

```python

# Create the script
nano XX.sh

# Submit the script
sbatch job.sh

squeue -u vawelke

scancel 1234567

tail -f logs/inj_J1852_g0_1234567_0.out

```